# Notebook 3: DPO Alignment Training
**Platform: Kaggle (2x T4 GPUs) — NEW notebook session**

## What this notebook does:
1. Loads SFT model from HuggingFace Hub (output of Notebook 2)
2. Loads DPO preference pairs from Google Drive (output of Notebook 1)
3. Runs DPO — teaches the model to prefer faithful + empathetic responses
4. Saves final aligned model to HuggingFace Hub

## Before running:
- Notebook 2 must be 100% complete
- Create a fresh Kaggle notebook (GPU T4 x2, Internet ON)
- Same HF_TOKEN secret as Notebook 2

In [1]:
!pip install -q transformers datasets peft trl accelerate bitsandbytes huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 540.5/540.5 kB 9.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.9 MB/s eta 0:00:00:00:0100:01


In [2]:
import os, json, torch
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secrets  = UserSecretsClient()
HF_TOKEN = secrets.get_secret('HF_TOKEN')
login(token=HF_TOKEN)

# REPLACE with your HuggingFace username
HF_USERNAME = 'Winindux'

BASE_MODEL = 'meta-llama/Llama-2-7b-chat-hf'
SFT_REPO   = f'{HF_USERNAME}/emowoz-llama2-sft'   # input  — from Notebook 2
DPO_REPO   = f'{HF_USERNAME}/emowoz-llama2-dpo'   # output — your final model
RESUME     = False

print(f'Loading SFT from : {SFT_REPO}')
print(f'Saving DPO to    : {DPO_REPO}')

Loading SFT from : Winindux/emowoz-llama2-sft
Saving DPO to    : Winindux/emowoz-llama2-dpo


In [3]:
# ── Notebook 3 data loading ────────────────────────────────────────
from huggingface_hub import hf_hub_download, login
from kaggle_secrets import UserSecretsClient
from datasets import Dataset
import json, os

secrets  = UserSecretsClient()
HF_TOKEN = secrets.get_secret('HF_TOKEN')
login(token=HF_TOKEN)

HF_USERNAME = 'Winindux'
DATA_REPO   = f'{HF_USERNAME}/emowoz-llama2-data'

dpo_path = hf_hub_download(
    repo_id=DATA_REPO,
    filename='data/dpo_pairs/dpo_pairs.jsonl',
    repo_type='dataset',
    local_dir='/kaggle/working',
    token=HF_TOKEN
)

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f]

dpo_raw   = load_jsonl(dpo_path)
dpo_ds    = Dataset.from_list([
    {'prompt': p['prompt'], 'chosen': p['chosen'], 'rejected': p['rejected']}
    for p in dpo_raw
])
dpo_split = dpo_ds.train_test_split(test_size=0.1, seed=42)
dpo_train = dpo_split['train']
dpo_val   = dpo_split['test']

print(f'DPO Train: {len(dpo_train)} | DPO Val: {len(dpo_val)}')

dpo_pairs.jsonl:   0%|          | 0.00/12.4k [00:00<?, ?B/s]

DPO Train: 7 | DPO Val: 1


In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16
)

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print('Loading base model (4-bit)...')
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    token=HF_TOKEN,
    torch_dtype=torch.bfloat16
)

print('Loading SFT DoRA adapter from HuggingFace Hub...')
# is_trainable=True makes this the policy model that DPO will update
model = PeftModel.from_pretrained(base, SFT_REPO, is_trainable=True)
model = prepare_model_for_kbit_training(model)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable/1e6:.1f}M / {total/1e6:.1f}M params')

Loading tokenizer...


config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

Loading SFT DoRA adapter from HuggingFace Hub...


adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/645M [00:00<?, ?B/s]

Trainable: 0.0M / 3661.7M params


In [5]:
# from trl import DPOTrainer, DPOConfig

# OUTPUT_DIR = '/kaggle/working/dpo_checkpoints'
# os.makedirs(OUTPUT_DIR, exist_ok=True)

# dpo_config = DPOConfig(
#     output_dir=OUTPUT_DIR,

#     # DPO-specific settings
#     beta=0.1,              # KL penalty — 0.1 is well-validated standard value
#                            # Lower = model drifts more toward preferences
#                            # Higher = model stays closer to SFT behavior
#     loss_type='sigmoid',   # standard DPO loss function

#     # Training
#     num_train_epochs=2,    # DPO needs fewer epochs than SFT
#     per_device_train_batch_size=1,   # smaller: DPO processes chosen+rejected together
#     per_device_eval_batch_size=1,
#     gradient_accumulation_steps=16,

#     # Memory
#     gradient_checkpointing=True,
#     optim='paged_adamw_8bit',
#     bf16=True,

#     # Learning rate — lower than SFT to avoid overwriting SFT knowledge
#     learning_rate=5e-5,
#     lr_scheduler_type='cosine',
#     warmup_steps=20,           # warmup_ratio removed in TRL v5.2+
#     max_grad_norm=0.3,

#     # Sequence
#     max_length=1024,
#     # max_prompt_length=512,

#     # Evaluation
#     eval_strategy='steps',
#     eval_steps=50,

#     # Checkpointing
#     save_strategy='steps',
#     save_steps=100,
#     save_total_limit=3,
#     load_best_model_at_end=True,

#     # Push to HF Hub
#     push_to_hub=True,
#     hub_model_id=DPO_REPO,
#     hub_strategy='checkpoint',
#     hub_token=HF_TOKEN,

#     logging_steps=25,
#     report_to='tensorboard',
# )

In [6]:
from trl import DPOTrainer, DPOConfig

OUTPUT_DIR = '/kaggle/working/dpo_checkpoints'
os.makedirs(OUTPUT_DIR, exist_ok=True)

dpo_config = DPOConfig(
    output_dir=OUTPUT_DIR,

    # DPO-specific settings
    beta=0.1,
    loss_type='sigmoid',

    # Training
    num_train_epochs=2,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,

    # Memory — gradient_checkpointing DISABLED for DoRA+DPO compatibility
    gradient_checkpointing=False,
    optim='paged_adamw_8bit',
    bf16=True,

    # Learning rate
    learning_rate=5e-5,
    lr_scheduler_type='cosine',
    warmup_steps=20,
    max_grad_norm=0.3,

    # Sequence
    max_length=1024,

    # Evaluation
    eval_strategy='steps',
    eval_steps=50,

    # Checkpointing
    save_strategy='steps',
    save_steps=100,
    save_total_limit=3,
    load_best_model_at_end=True,

    # Push to HF Hub
    push_to_hub=True,
    hub_model_id=DPO_REPO,
    hub_strategy='checkpoint',
    hub_token=HF_TOKEN,

    logging_steps=25,
    report_to='tensorboard',
)

In [7]:
# # Save final aligned adapter
# ADAPTER_DIR = '/kaggle/working/dpo_final_adapter'
# model.save_pretrained(ADAPTER_DIR)
# tokenizer.save_pretrained(ADAPTER_DIR)

# from huggingface_hub import HfApi
# api = HfApi()
# api.create_repo(DPO_REPO, private=True, exist_ok=True)
# api.upload_folder(
#     folder_path=ADAPTER_DIR,
#     repo_id=DPO_REPO,
#     commit_message='Final DPO aligned adapter — LLaMA 2 7B EmoWOZ'
# )
# print(f'Final model saved: https://huggingface.co/{DPO_REPO}')

# # Test with two examples — one anxious, one dissatisfied
# test_cases = [
#     ('anxious',       'The hotel is called The Gonville. It is in the city centre. Price: expensive. Phone: 01223366611.',
#                       "I need a hotel but I'm really stressed about the cost"),
#     ('dissatisfied',  'The hotel has free WiFi. No swimming pool available. Check-out at 11am.',
#                       'I was really hoping for a pool. Do you have one?'),
# ]
# SYSTEM = 'You are a compassionate assistant. Use ONLY the provided Facts. Match the user\'s emotional tone.'

# for emotion, facts, inquiry in test_cases:
#     prompt = (
#         f"<s>[INST] <<SYS>>\n{SYSTEM}\n<</SYS>>\n\n"
#         f"Emotional State: {emotion}\nFacts: {facts}\n"
#         f"Conversation History:\nNo previous turns.\nUser: {inquiry} [/INST]"
#     )
#     inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
#     with torch.no_grad():
#         out = model.generate(**inputs, max_new_tokens=120, temperature=0.7,
#                              do_sample=True, pad_token_id=tokenizer.eos_token_id)
#     resp = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
#     print(f'\n[{emotion}] {inquiry}')
#     print(f'→ {resp}')
#     print('-' * 60)

In [ ]:
# dpo_trainer = DPOTrainer(
#     model=model,
#     args=dpo_config,
#     train_dataset=dpo_train,
#     eval_dataset=dpo_val,
#     tokenizer=tokenizer
#     # ref_model=None: TRL uses the frozen initial adapter weights as reference
#     # This is memory-efficient and correct for PEFT-based DPO
# )

# if RESUME:
#     print('Resuming DPO from checkpoint...')
#     dpo_trainer.train(resume_from_checkpoint=True)
# else:
#     print('Starting DPO training...')
#     dpo_trainer.train()

# print('DPO training complete!')



dpo_trainer = DPOTrainer(
    model=model,
    args=dpo_config,
    train_dataset=dpo_train,
    eval_dataset=dpo_val,
    processing_class=tokenizer,    # tokenizer= renamed in TRL v5.2+
)

if RESUME:
    print('Resuming DPO from checkpoint...')
    dpo_trainer.train(resume_from_checkpoint=True)
else:
    print('Starting DPO training...')
    dpo_trainer.train()

print('DPO training complete!')


Extracting prompt in train dataset:   0%|          | 0/7 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/7 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/7 [00:00<?, ? examples/s]

Extracting prompt in eval dataset:   0%|          | 0/1 [00:00<?, ? examples/s]

Applying chat template to eval dataset:   0%|          | 0/1 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/1 [00:00<?, ? examples/s]

2026-02-22 18:03:50.942956: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771783431.108291      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771783431.157576      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771783431.554844      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771783431.554874      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771783431.554877      55 computation_placer.cc:177] computation placer alr

Starting DPO training...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


In [9]:
# Save final aligned adapter
ADAPTER_DIR = '/kaggle/working/dpo_final_adapter'
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

from huggingface_hub import HfApi
api = HfApi()
api.create_repo(DPO_REPO, private=True, exist_ok=True)
api.upload_folder(
    folder_path=ADAPTER_DIR,
    repo_id=DPO_REPO,
    commit_message='Final DPO aligned adapter — LLaMA 2 7B EmoWOZ'
)
print(f'Final model saved: https://huggingface.co/{DPO_REPO}')

# Test with two examples — one anxious, one dissatisfied
test_cases = [
    ('anxious',       'The hotel is called The Gonville. It is in the city centre. Price: expensive. Phone: 01223366611.',
                      "I need a hotel but I'm really stressed about the cost"),
    ('dissatisfied',  'The hotel has free WiFi. No swimming pool available. Check-out at 11am.',
                      'I was really hoping for a pool. Do you have one?'),
]
SYSTEM = 'You are a compassionate assistant. Use ONLY the provided Facts. Match the user\'s emotional tone.'

for emotion, facts, inquiry in test_cases:
    prompt = (
        f"<s>[INST] <<SYS>>\n{SYSTEM}\n<</SYS>>\n\n"
        f"Emotional State: {emotion}\nFacts: {facts}\n"
        f"Conversation History:\nNo previous turns.\nUser: {inquiry} [/INST]"
    )
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=120, temperature=0.7,
                             do_sample=True, pad_token_id=tokenizer.eos_token_id)
    resp = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f'\n[{emotion}] {inquiry}')
    print(f'→ {resp}')
    print('-' * 60)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
Caching is incompatible with gradient checkpointing in LlamaDecoderLayer. Setting `past_key_values=None`.


Final model saved: https://huggingface.co/Winindux/emowoz-llama2-dpo

[anxious] I need a hotel but I'm really stressed about the cost
→  brings everybody everybody nobody Begriffe Hinweis фев hopefully hopefully hopefully everyone➖ hopefully everybody everybody paździer sierp nobody obviously➖ Hinweis everybody Hinweis Hinweis everybody Jahrh Einzeln nobody obviously everybody everybody everybody фев фев nobody nobody➖ obviously Hinweis geprüft everybody hopefully eventually фев sierp➖ hopefully everybody фев Hinweisего фев Einzeln Unterscheidung фев Einzeln everybody червня nobody Unterscheidung Unterscheidung nobody everybody hopefully everyone Einzeln живело surely obviously фев Einzeln nobody округу nobody hopefully nobody nobody nobody surelyϊ nobody фев everybody Hinweis nobody Unterscheidungἱ paździer obviously everybody everybody obviously hopefully nobody Hinweis Hinweisϊ фев червня everybody kwiet Einzeln hopefully obviously Hinweisℓ obviously hopefully живело Unterscheidungϊ

## ✅ Notebook 3 Complete!
Final model at: `huggingface.co/YOUR_USERNAME/emowoz-llama2-dpo`

## ➡️ Next: Go back to Google Colab and run Notebook 4 (Evaluation)